In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Global Constants ---
ARTIFACTS_DIR = 'Scenario 1' # Reverted to old name
MODEL_FILENAME = 'random_forest_regressor_model_sales.joblib' # Reverted to .joblib for RF
LABEL_ENCODERS_FILENAME = 'label_encoders_sales_value.joblib'
PRODUCT_AVG_FEATURES_MAP_FILENAME = 'product_avg_features_map.csv'
ARTIFACTS_INFO_FILENAME = 'artifacts_info_sales_value.joblib'

FEATURE_COLUMNS = ['Category', 'Sub-Category', 'Product Name',
                   'Average_Price_Per_Unit', 'Average_Discount', 'Average_Profit_Per_Sale',
                   'Month_sin', 'Month_cos'] # Reverted: Removed 'Customer_Increase_Percentage'
TARGET_COLUMN = 'MonthlySales'

# Categorical columns that need Label Encoding (remains the same)
CATEGORICAL_FEATURES = ['Category', 'Sub-Category', 'Product Name']


In [ ]:
# --- 0. generate_synthetic_sales_data (MODIFIED - Reverted and added seasonality parameters) ---
def generate_synthetic_sales_data(
    n_rows=10000, # Increased rows for better training
    seed=42, 
    pct_no_purchase=0.4,
    start_year=2015, # Reverted start year to earlier for more data
    end_year=2024,   # Current year
    peak_month=12,   # Month with highest sales (e.g., 12 for December)
    seasonality_amplitude=0.3, # How strong seasonality is (e.g., +/- 30%)
    yearly_growth_rate=0.03 # Annual sales growth (e.g., 3% per year)
):
    np.random.seed(seed)
    random.seed(seed)
    if fake is None:
        print("Faker not initialized. Cannot generate some synthetic data attributes.")
        return pd.DataFrame()

    ship_modes = ['First Class', 'Second Class', 'Standard Class', 'Same Day']
    segments = ['Consumer', 'Corporate', 'Home Office']
    categories = ['Furniture', 'Office Supplies', 'Technology']
    sub_categories = {
        'Furniture': ['Chairs', 'Tables', 'Bookcases'],
        'Office Supplies': ['Binders', 'Paper', 'Art'],
        'Technology': ['Phones', 'Accessories', 'Copiers']
    }
    products = {
        'Chairs': ['Ergonomic Chair', 'Mesh Chair'],
        'Tables': ['Conference Table', 'Office Desk'],
        'Bookcases': ['Wooden Bookcase', 'Metal Bookcase'],
        'Binders': ['3-Ring Binder', 'Heavy-Duty Binder'],
        'Paper': ['Printer Paper', 'Notebook Paper'],
        'Art': ['Markers', 'Sketch Pens'],
        'Phones': ['iPhone 14', 'Samsung Galaxy'],
        'Accessories': ['Mouse', 'Keyboard'],
        'Copiers': ['Canon Copier', 'HP Copier']
    }

    countries_by_region = {
        'North America': ['United States', 'Canada', 'Mexico'],
        'Europe': ['United Kingdom', 'Germany', 'France', 'Italy', 'Spain', 'Netherlands', 'Sweden'],
        'Asia': ['India', 'China', 'Japan', 'South Korea', 'Singapore', 'Indonesia'],
        'South America': ['Brazil', 'Argentina', 'Chile', 'Colombia'],
        'Africa': ['South Africa', 'Nigeria', 'Egypt', 'Kenya'],
        'Oceania': ['Australia', 'New Zealand']
    }
    all_countries = [country for region in countries_by_region.values() for country in region]

    avg_rows_per_customer = 100.8
    num_customers = int(np.ceil(n_rows / avg_rows_per_customer))

    customers = []
    used_customer_ids = set()
    while len(customers) < num_customers:
        cust_id = f"{random.choice(['CG', 'SC', 'NG'])}-{random.randint(10000, 99999)}"
        if cust_id in used_customer_ids:
            continue
        used_customer_ids.add(cust_id)
        cust_name = fake.name()
        segment = random.choice(segments)
        country = random.choice(all_countries)
        region = [r for r, countries in countries_by_region.items() if country in countries][0]
        state = fake.state()
        city = fake.city()
        postal_code = fake.postcode()

        customers.append({
            'Customer ID': cust_id,
            'Customer Name': cust_name,
            'Segment': segment,
            'Country': country,
            'Region': region,
            'State': state,
            'City': city,
            'Postal Code': postal_code
        })
    customers_df = pd.DataFrame(customers)

    customer_indices = np.repeat(range(num_customers), int(avg_rows_per_customer))
    while len(customer_indices) < n_rows:
        customer_indices = np.concatenate([customer_indices, np.repeat(range(num_customers), int(avg_rows_per_customer))])
    customer_indices = customer_indices[:n_rows]
    np.random.shuffle(customer_indices)

    n_no_purchase = int(n_rows * pct_no_purchase)
    no_purchase_indices = set(random.sample(range(n_rows), n_no_purchase))

    data = []

    for i in range(n_rows):
        cust = customers_df.iloc[customer_indices[i]]

        start_date_dt = datetime(start_year, 1, 1)
        end_date_dt = datetime(end_year, 12, 31)
        time_delta = end_date_dt - start_date_dt
        order_date = start_date_dt + timedelta(days=random.randint(0, time_delta.days))

        ship_date = order_date + timedelta(days=np.random.randint(1, 7))
        ship_mode = random.choice(ship_modes)

        category = random.choice(categories)
        sub_category = random.choice(sub_categories[category])
        product_name = random.choice(products[sub_category])

        month_num = order_date.month
        year_num = order_date.year
        
        seasonality_factor = 1.0 + seasonality_amplitude * np.cos(2 * np.pi * (month_num - peak_month) / 12)
        years_since_start = year_num - start_year
        growth_factor = 1.0 + (years_since_start * yearly_growth_rate)

        if i in no_purchase_indices:
            quantity = 0
            sales = 0.0
            profit = 0.0
            discount = 0.0
        else:
            quantity = np.random.randint(1, 10)
            base_price = np.random.uniform(20, 500)
            discount = round(np.random.choice([0.0, 0.1, 0.2, 0.3]), 2)
            
            raw_sales = quantity * base_price * (1 - discount)
            
            sales = round(raw_sales * seasonality_factor * growth_factor, 2)
            
            quantity = max(1, round(quantity * seasonality_factor * growth_factor * np.random.uniform(0.9, 1.1)))

            profit = round(sales * np.random.uniform(0.05, 0.3), 2)

        country_code = ''.join([c[0] for c in cust['Country'].split()])[:2].upper()
        order_id = f"{country_code}-{order_date.year}-{random.randint(100000, 999999)}"
        product_id = f"{category[:3].upper()}-{sub_category[:2].upper()}-{random.randint(10000000, 99999999)}"

        data.append([
            i + 1,
            order_id,
            order_date.strftime('%Y-%m-%d'),
            ship_date.strftime('%Y-%m-%d'),
            ship_mode,
            cust['Customer ID'],
            cust['Customer Name'],
            cust['Segment'],
            cust['Country'],
            cust['City'],
            cust['State'],
            cust['Postal Code'],
            cust['Region'],
            product_id,
            category,
            sub_category,
            product_name,
            round(sales, 2),
            quantity,
            discount,
            round(profit, 2)
        ])

    columns = ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
               'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
               'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
               'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

    print(f"Generated {n_rows} rows of synthetic data.")
    return pd.DataFrame(data, columns=columns)


# --- 1. prepare_data (REVERTED) ---
def prepare_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("Preparing data for total sales value prediction with average numerical features and Fourier time features...")
    df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed')
    df['YearMonth'] = df['Order Date'].dt.to_period('M').astype(str)
    df['Price_Per_Unit_Transaction'] = df.apply(lambda row: row['Sales'] / row['Quantity'] if row['Quantity'] > 0 else 0, axis=1)

    product_avg_features = df.groupby(['Category', 'Sub-Category', 'Product Name']).agg(
        Average_Price_Per_Unit=('Price_Per_Unit_Transaction', 'mean'),
        Average_Discount=('Discount', 'mean'),
        Average_Profit_Per_Sale=('Profit', 'mean')
    ).reset_index()
    product_avg_features = product_avg_features.fillna(0)

    monthly_product_sales = df.groupby(['Product ID', 'Category', 'Sub-Category', 'Product Name', 'YearMonth'])['Sales'].sum().reset_index()
    monthly_product_sales.rename(columns={'Sales': 'MonthlySales'}, inplace=True)
    
    monthly_product_sales['Month_of_Year_Numeric'] = pd.to_datetime(monthly_product_sales['YearMonth']).dt.month
    monthly_product_sales['Month_sin'] = np.sin(2 * np.pi * monthly_product_sales['Month_of_Year_Numeric'] / 12)
    monthly_product_sales['Month_cos'] = np.cos(2 * np.pi * monthly_product_sales['Month_of_Year_Numeric'] / 12)

    processed_df = pd.merge(monthly_product_sales, product_avg_features,
                            on=['Category', 'Sub-Category', 'Product Name'],
                            how='left')

    for col in CATEGORICAL_FEATURES: # Use CATEGORICAL_FEATURES global
        if col in processed_df.columns:
            processed_df[col] = processed_df[col].astype('category')

    print("Data preparation complete.")
    return processed_df, product_avg_features

# --- 2. encode (REVERTED - No Numerical Scaling) ---
def encode(processed_df: pd.DataFrame) -> tuple[pd.DataFrame, dict]: # Reverted return type
    print("Encoding data...")
    X = processed_df[FEATURE_COLUMNS].copy() 
    categorical_features_to_encode = [col for col in FEATURE_COLUMNS if X[col].dtype in ['object', 'category']]
    
    label_encoders = {}
    for col in categorical_features_to_encode:
        le = LabelEncoder()
        X[col] = X[col].astype(str) 
        le.fit(X[col].unique())
        X[col] = le.transform(X[col])
        label_encoders[col] = le
        
    print("Data encoding complete.")
    return X, label_encoders # Reverted return

# --- 3. build_model (REVERTED - Random Forest) ---
def build_model():
    print("Building Random Forest Regressor model...")
    model = RandomForestRegressor(random_state=42, n_estimators=100)
    print("Model built.")
    return model

# --- 4. train_model (REVERTED - Random Forest Training) ---
def train_model(model: RandomForestRegressor, X_train: pd.DataFrame, y_train: pd.Series) -> RandomForestRegressor:
    print("Training Random Forest Regressor model...")
    for col in X_train.columns:
        if X_train[col].dtype == 'object':
            raise TypeError(f"Column '{col}' is still of 'object' dtype (contains strings). All features must be numerical for training.")
            
    model.fit(X_train, y_train)
    print("Model training complete.")
    return model

# --- 5. evaluate_model (REVERTED - Random Forest Evaluation) ---
def evaluate_model(model: RandomForestRegressor, X_test: pd.DataFrame, y_test: pd.Series):
    print("Evaluating model...")
    y_pred = model.predict(X_test)
    print("\n--- Model Evaluation Report (Regression) ---")
    print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, y_pred):.2f}")
    print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
    print(f"R-squared (R2): {r2_score(y_test, y_pred):.4f}")
    print("-------------------------------------------\n")

# --- 6. save_artifacts (REVERTED - Random Forest Model, no Scaler) ---
def save_artifacts(model: RandomForestRegressor, label_encoders: dict, feature_columns: list, 
                   target_column: str, product_avg_features_map: pd.DataFrame, artifacts_dir: str = ARTIFACTS_DIR):
    print(f"Saving artifacts to '{artifacts_dir}'...")
    os.makedirs(artifacts_dir, exist_ok=True) 

    joblib.dump(model, os.path.join(artifacts_dir, MODEL_FILENAME)) # Reverted model save
    joblib.dump(label_encoders, os.path.join(artifacts_dir, LABEL_ENCODERS_FILENAME))
    
    artifacts_info = {
        'feature_columns': feature_columns,
        'target_column': target_column
    }
    joblib.dump(artifacts_info, os.path.join(artifacts_dir, ARTIFACTS_INFO_FILENAME))
    
    product_avg_features_map.to_csv(os.path.join(artifacts_dir, PRODUCT_AVG_FEATURES_MAP_FILENAME), index=False)

    print("Artifacts saved successfully.")

# --- 7. load_artifacts (REVERTED - Random Forest Model, no Scaler) ---
def load_artifacts(artifacts_dir: str = ARTIFACTS_DIR) -> tuple:
    """
    Loads the trained model, label encoders, feature columns, target column,
    and the product average features map from the specified directory.
    
    Args:
        artifacts_dir (str): Path to the directory where artifacts are saved.
        
    Returns:
        tuple: (loaded_model, loaded_label_encoders, loaded_feature_columns,
                loaded_target_column, loaded_product_avg_features_map)
    """
    print(f"Loading artifacts from '{artifacts_dir}'...")
    try:
        loaded_model = joblib.load(os.path.join(artifacts_dir, MODEL_FILENAME)) # Reverted model load
        loaded_label_encoders = joblib.load(os.path.join(artifacts_dir, LABEL_ENCODERS_FILENAME))
        artifacts_info = joblib.load(os.path.join(artifacts_dir, ARTIFACTS_INFO_FILENAME))
        loaded_feature_columns = artifacts_info['feature_columns']
        loaded_target_column = artifacts_info['target_column']
        
        loaded_product_avg_features_map = pd.read_csv(os.path.join(artifacts_dir, PRODUCT_AVG_FEATURES_MAP_FILENAME))
        
        print("Artifacts loaded successfully.")
        return loaded_model, loaded_label_encoders, loaded_feature_columns, \
               loaded_target_column, loaded_product_avg_features_map
    except FileNotFoundError as e:
        raise FileNotFoundError(f"One or more artifact files not found in {artifacts_dir}. Error: {e}")
    except Exception as e:
        raise Exception(f"An error occurred while loading artifacts: {e}")

# --- 8. predictions (REVERTED to previous flexible DataFrame/Dict output) ---
def predictions(
    forecast_period: int, 
    loaded_model, 
    loaded_label_encoders, 
    loaded_feature_columns, 
    loaded_product_avg_features_map,
    category: str = None, # Made optional
    sub_category: str = None, # Made optional
    product_name: str = None # Made optional
) -> [pd.DataFrame, dict]: # Indicate possible return types
    """
    Predicts the total sales value for specific product combinations for the next few months,
    allowing for flexible input on product features (Category, Sub-Category, Product Name).

    Args:
        forecast_period (int): Number of months to forecast.
        loaded_model (RandomForestRegressor): The trained regression model.
        loaded_label_encoders (dict): Dictionary of fitted LabelEncoders.
        loaded_feature_columns (list): List of feature column names used during training.
        loaded_product_avg_features_map (pd.DataFrame): DataFrame containing average numerical features.
        category (str, optional): Specific Category to filter predictions. Defaults to None.
        sub_category (str, optional): Specific Sub-Category to filter predictions. Defaults to None.
        product_name (str, optional): Specific Product Name to filter predictions. Defaults to None.

    Returns:
        pd.DataFrame: If multiple product combinations are predicted (Category, Sub-Category, Product Name
                      are not all specified). Contains columns for product details, forecast month, and predicted sales.
        dict: If a single, specific product (Category, Sub-Category, AND Product Name all specified)
              is requested. Contains forecast months as keys and predicted sales values (float) as values.
    """
    print("Generating total sales value predictions with flexible product filtering...")
    
    current_time_for_forecast = datetime.now()
    all_raw_predictions = [] # List to collect all individual month-product predictions

    # --- Determine the product combinations to predict for ---
    target_products_df = loaded_product_avg_features_map.copy()

    if category:
        target_products_df = target_products_df[target_products_df['Category'] == category]
    if sub_category:
        target_products_df = target_products_df[target_products_df['Sub-Category'] == sub_category]
    if product_name:
        target_products_df = target_products_df[target_products_df['Product Name'] == product_name]

    if target_products_df.empty:
        print("No product combinations found matching the provided criteria. Returning empty output.")
        # Return an empty DataFrame or dictionary based on the expected scenario
        if category and sub_category and product_name:
            return {} # If looking for a specific product, return empty dict
        else:
            return pd.DataFrame(columns=['Category', 'Sub-Category', 'Product Name', 'Forecast_Month', 'Predicted_Sales']) # Otherwise, empty DataFrame

    # Flag to determine final return type
    return_single_product_dict = (category is not None and sub_category is not None and product_name is not None and len(target_products_df) == 1)
    
    # If we are returning a single product dictionary, store its predictions here temporarily
    single_product_forecasts = {}

    # Iterate through each unique product combination found
    for idx, product_row in target_products_df.iterrows():
        # Extract product details and average features for the current product combination
        current_product_category = product_row['Category']
        current_product_sub_category = product_row['Sub-Category']
        current_product_name = product_row['Product Name']

        avg_price = product_row['Average_Price_Per_Unit']
        avg_discount = product_row['Average_Discount']
        avg_profit = product_row['Average_Profit_Per_Sale']

        for i in range(forecast_period):
            # Determine the forecast month and extract its numeric month (1-12)
            forecast_date = current_time_for_forecast + timedelta(days=30 * (i + 1))
            forecast_month_str = forecast_date.strftime('%Y-%m')
            forecast_month_num = forecast_date.month 

            # Calculate sine and cosine components for the forecast month
            month_sin = np.sin(2 * np.pi * forecast_month_num / 12)
            month_cos = np.cos(2 * np.pi * forecast_month_num / 12)

            # Create the input row for prediction
            current_input_data = {
                'Category': current_product_category,
                'Sub-Category': current_product_sub_category,
                'Product Name': current_product_name,
                'Average_Price_Per_Unit': avg_price,
                'Average_Discount': avg_discount,
                'Average_Profit_Per_Sale': avg_profit,
                'Month_sin': month_sin,
                'Month_cos': month_cos
            }
            input_df_for_prediction = pd.DataFrame([current_input_data])
            
            # Ensure correct column order and types for prediction
            input_df_for_prediction = input_df_for_prediction.reindex(columns=loaded_feature_columns, fill_value=np.nan)

            # Apply label encoding to categorical columns
            for col in CATEGORICAL_FEATURES: # Use CATEGORICAL_FEATURES global
                if col in loaded_label_encoders:
                    le = loaded_label_encoders[col]
                    val_to_transform = str(input_df_for_prediction[col].iloc[0]) 
                    
                    if val_to_transform in le.classes_:
                        input_df_for_prediction[col] = le.transform([val_to_transform])[0]
                    else:
                        # Handle unseen categories/sub-categories/products
                        unknown_id = len(le.classes_) 
                        input_df_for_prediction[col] = unknown_id
                        print(f"Warning: Value '{val_to_transform}' in column '{col}' not seen during training. Assigned unknown ID: {unknown_id}.")
            
            predicted_sales_value = loaded_model.predict(input_df_for_prediction)[0]
            
            if return_single_product_dict:
                single_product_forecasts[forecast_month_str] = round(predicted_sales_value, 2)
            else:
                # For DataFrame output, append a row for each product-month prediction
                all_raw_predictions.append({
                    'Category': current_product_category,
                    'Sub-Category': current_product_sub_category,
                    'Product Name': current_product_name,
                    'Forecast_Month': forecast_month_str,
                    'Predicted_Sales': round(predicted_sales_value, 2)
                })
        
    print("Total sales value predictions complete.")

    if return_single_product_dict:
        return single_product_forecasts
    else:
        return pd.DataFrame(all_raw_predictions)



In [30]:

# --- Main execution block for testing the pipeline ---
if __name__ == "__main__":
    print("\n--- Starting ML Workflow for Total Sales Value Prediction (Random Forest) ---")

    # Clean up previous artifacts if they exist to avoid conflicts
    if os.path.exists(ARTIFACTS_DIR):
        print(f"Cleaning up existing artifact directory: {ARTIFACTS_DIR}")
        import shutil
        shutil.rmtree(ARTIFACTS_DIR)

    # 1. Generate synthetic data with seasonality
    print("\n--- Loading Raw Data ---")
    raw_df = pd.read_csv(f'{os.getcwd()}/../data/raw_data/Complete.csv')
    # print(raw_df.head())
    # print(raw_df.info())

    # 2. Prepare Data
    print("\n--- Preparing Data ---")
    processed_data, product_avg_features_map_for_save = prepare_data(raw_df.copy())
    # print(processed_data.head())
    # print(processed_data.info())


    # 3. Encode Data
    print("\n--- Encoding Data ---")
    X_encoded, label_encoders_fitted = encode(processed_data.copy())
    y_target = processed_data[TARGET_COLUMN]

    # Split data for training and evaluation
    X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
        X_encoded, y_target, test_size=0.2, random_state=42
    )

    # 4. Build Model
    print("\n--- Building Model ---")
    ml_model = build_model()

    # 5. Train Model
    print("\n--- Training Model ---")
    trained_ml_model = train_model(ml_model, X_train_split, y_train_split)

    # 6. Evaluate Model
    print("\n--- Evaluating Model ---")
    evaluate_model(trained_ml_model, X_test_split, y_test_split)

    # 7. Save Artifacts
    print("\n--- Saving Artifacts ---")
    save_artifacts(trained_ml_model, label_encoders_fitted, FEATURE_COLUMNS, TARGET_COLUMN, product_avg_features_map_for_save)

    # # --- Demonstrating Flexible Prediction with Loaded Artifacts ---
    # print("\n--- Demonstrating Flexible Prediction with Loaded Artifacts ---")

    # # 8. Load Artifacts
    # loaded_model, loaded_encoders, loaded_feature_cols, loaded_target_col, loaded_product_avg_map = load_artifacts()

    # # Scenario 1: Predict for ALL products (should return DataFrame)
    # print("\nScenario 1: Predicting for ALL products for next 3 months (Expected: DataFrame)")
    # predictions_all_products = predictions(
    #     forecast_period=3,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map
    # )
    # print(predictions_all_products.head())
    # print(f"Shape: {predictions_all_products.shape}, Type: {type(predictions_all_products)}")


    # # Scenario 2: Predict for a specific Category (should return DataFrame)
    # print("\nScenario 2: Predicting for 'Electronics' category for next 2 months (Expected: DataFrame)")
    # predictions_category = predictions(
    #     forecast_period=2,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map,
    #     category='Technology' # Changed to Technology for common synthetic data
    # )
    # print(predictions_category.head())
    # print(f"Shape: {predictions_category.shape}, Type: {type(predictions_category)}")


    # # Scenario 3: Predict for a specific Category and Sub-Category (should return DataFrame)
    # print("\nScenario 3: Predicting for 'Office Supplies' -> 'Paper' for next 1 month (Expected: DataFrame)")
    # predictions_sub_category = predictions(
    #     forecast_period=1,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map,
    #     category='Office Supplies',
    #     sub_category='Paper'
    # )
    # print(predictions_sub_category.head())
    # print(f"Shape: {predictions_sub_category.shape}, Type: {type(predictions_sub_category)}")


    # # Scenario 4: Predict for a specific Category, Sub-Category, and Product Name (should return Dictionary)
    # print("\nScenario 4: Predicting for 'Technology' -> 'Phones' -> 'iPhone 14' for next 3 months (Expected: Dictionary)")
    # predictions_specific_product = predictions(
    #     forecast_period=3,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map,
    #     category='Technology',
    #     sub_category='Phones',
    #     product_name='iPhone 14'
    # )
    # print(predictions_specific_product)
    # print(f"Type: {type(predictions_specific_product)}")

    # # Scenario 5: Product not found (should return empty DataFrame or dict as per logic)
    # print("\nScenario 5: Predicting for a non-existent product (Expected: Empty output based on previous type)")
    # predictions_not_found_specific = predictions(
    #     forecast_period=1,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map,
    #     category='NonExistentCategory',
    #     sub_category='NonExistentSubCategory',
    #     product_name='NonExistentProduct'
    # )
    # print(predictions_not_found_specific)
    # print(f"Type: {type(predictions_not_found_specific)}")

    # predictions_not_found_category = predictions(
    #     forecast_period=1,
    #     loaded_model=loaded_model,
    #     loaded_label_encoders=loaded_encoders,
    #     loaded_feature_columns=loaded_feature_cols,
    #     loaded_product_avg_features_map=loaded_product_avg_map,
    #     category='NonExistentCategory'
    # )
    # print(predictions_not_found_category)
    # print(f"Type: {type(predictions_not_found_category)}")


    # print("\n--- Flexible Forecasting Demonstration Complete ---")



--- Starting ML Workflow for Total Sales Value Prediction (Random Forest) ---

--- Loading Raw Data ---

--- Preparing Data ---
Preparing data for total sales value prediction with average numerical features and Fourier time features...
Data preparation complete.

--- Encoding Data ---
Encoding data...
Data encoding complete.

--- Building Model ---
Building Random Forest Regressor model...
Model built.

--- Training Model ---
Training Random Forest Regressor model...
Model training complete.

--- Evaluating Model ---
Evaluating model...

--- Model Evaluation Report (Regression) ---
Mean Absolute Error (MAE): 731.77
Mean Squared Error (MSE): 1032113.26
Root Mean Squared Error (RMSE): 1015.93
R-squared (R2): 0.0818
-------------------------------------------


--- Saving Artifacts ---
Saving artifacts to 'Scenario 1'...
Artifacts saved successfully.
Loading artifacts from 'Scenario 1'...
Artifacts loaded successfully.

Scenario 1: Predicting for ALL products for next 3 months (Expected

In [3]:

# --- 7. load_artifacts (REVERTED - Random Forest Model, no Scaler) ---
def load_artifacts(artifacts_dir: str = ARTIFACTS_DIR) -> tuple:
    """
    Loads the trained model, label encoders, feature columns, target column,
    and the product average features map from the specified directory.
    
    Args:
        artifacts_dir (str): Path to the directory where artifacts are saved.
        
    Returns:
        tuple: (loaded_model, loaded_label_encoders, loaded_feature_columns,
                loaded_target_column, loaded_product_avg_features_map)
    """
    print(f"Loading artifacts from '{artifacts_dir}'...")
    try:
        loaded_model = joblib.load(os.path.join(artifacts_dir, MODEL_FILENAME)) # Reverted model load
        loaded_label_encoders = joblib.load(os.path.join(artifacts_dir, LABEL_ENCODERS_FILENAME))
        artifacts_info = joblib.load(os.path.join(artifacts_dir, ARTIFACTS_INFO_FILENAME))
        loaded_feature_columns = artifacts_info['feature_columns']
        loaded_target_column = artifacts_info['target_column']
        
        loaded_product_avg_features_map = pd.read_csv(os.path.join(artifacts_dir, PRODUCT_AVG_FEATURES_MAP_FILENAME))
        
        print("Artifacts loaded successfully.")
        return loaded_model, loaded_label_encoders, loaded_feature_columns, \
               loaded_target_column, loaded_product_avg_features_map
    except FileNotFoundError as e:
        raise FileNotFoundError(f"One or more artifact files not found in {artifacts_dir}. Error: {e}")
    except Exception as e:
        raise Exception(f"An error occurred while loading artifacts: {e}")

# --- 8. predictions (REVERTED to previous flexible DataFrame/Dict output) ---
def predictions(
    forecast_period: int, 
    loaded_model, 
    loaded_label_encoders, 
    loaded_feature_columns, 
    loaded_product_avg_features_map,
    category: str = None, # Made optional
    sub_category: str = None, # Made optional
    product_name: str = None # Made optional
) -> [pd.DataFrame, dict]: # Indicate possible return types
    """
    Predicts the total sales value for specific product combinations for the next few months,
    allowing for flexible input on product features (Category, Sub-Category, Product Name).

    Args:
        forecast_period (int): Number of months to forecast.
        loaded_model (RandomForestRegressor): The trained regression model.
        loaded_label_encoders (dict): Dictionary of fitted LabelEncoders.
        loaded_feature_columns (list): List of feature column names used during training.
        loaded_product_avg_features_map (pd.DataFrame): DataFrame containing average numerical features.
        category (str, optional): Specific Category to filter predictions. Defaults to None.
        sub_category (str, optional): Specific Sub-Category to filter predictions. Defaults to None.
        product_name (str, optional): Specific Product Name to filter predictions. Defaults to None.

    Returns:
        pd.DataFrame: If multiple product combinations are predicted (Category, Sub-Category, Product Name
                      are not all specified). Contains columns for product details, forecast month, and predicted sales.
        dict: If a single, specific product (Category, Sub-Category, AND Product Name all specified)
              is requested. Contains forecast months as keys and predicted sales values (float) as values.
    """
    print("Generating total sales value predictions with flexible product filtering...")
    
    current_time_for_forecast = datetime.now()
    all_raw_predictions = [] # List to collect all individual month-product predictions

    # --- Determine the product combinations to predict for ---
    target_products_df = loaded_product_avg_features_map.copy()

    if category:
        target_products_df = target_products_df[target_products_df['Category'] == category]
    if sub_category:
        target_products_df = target_products_df[target_products_df['Sub-Category'] == sub_category]
    if product_name:
        target_products_df = target_products_df[target_products_df['Product Name'] == product_name]

    if target_products_df.empty:
        print("No product combinations found matching the provided criteria. Returning empty output.")
        # Return an empty DataFrame or dictionary based on the expected scenario
        if category and sub_category and product_name:
            return {} # If looking for a specific product, return empty dict
        else:
            return pd.DataFrame(columns=['Category', 'Sub-Category', 'Product Name', 'Forecast_Month', 'Predicted_Sales']) # Otherwise, empty DataFrame

    # Flag to determine final return type
    return_single_product_dict = (category is not None and sub_category is not None and product_name is not None and len(target_products_df) == 1)
    
    # If we are returning a single product dictionary, store its predictions here temporarily
    single_product_forecasts = {}

    # Iterate through each unique product combination found
    for idx, product_row in target_products_df.iterrows():
        # Extract product details and average features for the current product combination
        current_product_category = product_row['Category']
        current_product_sub_category = product_row['Sub-Category']
        current_product_name = product_row['Product Name']

        avg_price = product_row['Average_Price_Per_Unit']
        avg_discount = product_row['Average_Discount']
        avg_profit = product_row['Average_Profit_Per_Sale']

        for i in range(forecast_period):
            # Determine the forecast month and extract its numeric month (1-12)
            forecast_date = current_time_for_forecast + timedelta(days=30 * (i + 1))
            forecast_month_str = forecast_date.strftime('%Y-%m')
            forecast_month_num = forecast_date.month 

            # Calculate sine and cosine components for the forecast month
            month_sin = np.sin(2 * np.pi * forecast_month_num / 12)
            month_cos = np.cos(2 * np.pi * forecast_month_num / 12)

            # Create the input row for prediction
            current_input_data = {
                'Category': current_product_category,
                'Sub-Category': current_product_sub_category,
                'Product Name': current_product_name,
                'Average_Price_Per_Unit': avg_price,
                'Average_Discount': avg_discount,
                'Average_Profit_Per_Sale': avg_profit,
                'Month_sin': month_sin,
                'Month_cos': month_cos
            }
            input_df_for_prediction = pd.DataFrame([current_input_data])
            
            # Ensure correct column order and types for prediction
            input_df_for_prediction = input_df_for_prediction.reindex(columns=loaded_feature_columns, fill_value=np.nan)

            # Apply label encoding to categorical columns
            for col in CATEGORICAL_FEATURES: # Use CATEGORICAL_FEATURES global
                if col in loaded_label_encoders:
                    le = loaded_label_encoders[col]
                    val_to_transform = str(input_df_for_prediction[col].iloc[0]) 
                    
                    if val_to_transform in le.classes_:
                        input_df_for_prediction[col] = le.transform([val_to_transform])[0]
                    else:
                        # Handle unseen categories/sub-categories/products
                        unknown_id = len(le.classes_) 
                        input_df_for_prediction[col] = unknown_id
                        print(f"Warning: Value '{val_to_transform}' in column '{col}' not seen during training. Assigned unknown ID: {unknown_id}.")
            
            predicted_sales_value = loaded_model.predict(input_df_for_prediction)[0]
            
            if return_single_product_dict:
                single_product_forecasts[forecast_month_str] = round(predicted_sales_value, 2)
            else:
                # For DataFrame output, append a row for each product-month prediction
                all_raw_predictions.append({
                    'Category': current_product_category,
                    'Sub-Category': current_product_sub_category,
                    'Product Name': current_product_name,
                    'Forecast_Month': forecast_month_str,
                    'Predicted_Sales': round(predicted_sales_value, 2)
                })
        
    print("Total sales value predictions complete.")

    if return_single_product_dict:
        return single_product_forecasts
    else:
        return pd.DataFrame(all_raw_predictions)


In [5]:

# 8. Load Artifacts
loaded_model, loaded_encoders, loaded_feature_cols, loaded_target_col, loaded_product_avg_map = load_artifacts()

# # Scenario 1: Predict for ALL products (should return DataFrame)
# print("\nScenario 1: Predicting for ALL products for next 3 months (Expected: DataFrame)")
# predictions_all_products = predictions(
#     forecast_period=3,
#     loaded_model=loaded_model,
#     loaded_label_encoders=loaded_encoders,
#     loaded_feature_columns=loaded_feature_cols,
#     loaded_product_avg_features_map=loaded_product_avg_map
# )
# print(f"Shape: {predictions_all_products.shape}, Type: {type(predictions_all_products)}")
# predictions_all_products

# Scenario 2: Predict for a specific Category (should return DataFrame)
print("\nScenario 2: Predicting for 'Electronics' category for next 2 months (Expected: DataFrame)")
predictions_category = predictions(
    forecast_period=5,
    loaded_model=loaded_model,
    loaded_label_encoders=loaded_encoders,
    loaded_feature_columns=loaded_feature_cols,
    loaded_product_avg_features_map=loaded_product_avg_map,
    category='Technology' # Changed to Technology for common synthetic data
)
print(f"Shape: {predictions_category.shape}, Type: {type(predictions_category)}")
predictions_category

# # Scenario 3: Predict for a specific Category and Sub-Category (should return DataFrame)
# print("\nScenario 3: Predicting for 'Office Supplies' -> 'Paper' for next 1 month (Expected: DataFrame)")
# predictions_sub_category = predictions(
#     forecast_period=1,
#     loaded_model=loaded_model,
#     loaded_label_encoders=loaded_encoders,
#     loaded_feature_columns=loaded_feature_cols,
#     loaded_product_avg_features_map=loaded_product_avg_map,
#     category='Office Supplies',
#     sub_category='Paper'
# )
# print(predictions_sub_category.head())
# print(f"Shape: {predictions_sub_category.shape}, Type: {type(predictions_sub_category)}")


# # Scenario 4: Predict for a specific Category, Sub-Category, and Product Name (should return Dictionary)
# print("\nScenario 4: Predicting for 'Technology' -> 'Phones' -> 'iPhone 14' for next 3 months (Expected: Dictionary)")
# predictions_specific_product = predictions(
#     forecast_period=3,
#     loaded_model=loaded_model,
#     loaded_label_encoders=loaded_encoders,
#     loaded_feature_columns=loaded_feature_cols,
#     loaded_product_avg_features_map=loaded_product_avg_map,
#     category='Technology',
#     sub_category='Phones',
#     product_name='iPhone 14'
# )
# print(predictions_specific_product)
# print(f"Type: {type(predictions_specific_product)}")

# # Scenario 5: Product not found (should return empty DataFrame or dict as per logic)
# print("\nScenario 5: Predicting for a non-existent product (Expected: Empty output based on previous type)")
# predictions_not_found_specific = predictions(
#     forecast_period=1,
#     loaded_model=loaded_model,
#     loaded_label_encoders=loaded_encoders,
#     loaded_feature_columns=loaded_feature_cols,
#     loaded_product_avg_features_map=loaded_product_avg_map,
#     category='NonExistentCategory',
#     sub_category='NonExistentSubCategory',
#     product_name='NonExistentProduct'
# )
# print(predictions_not_found_specific)
# print(f"Type: {type(predictions_not_found_specific)}")

# predictions_not_found_category = predictions(
#     forecast_period=1,
#     loaded_model=loaded_model,
#     loaded_label_encoders=loaded_encoders,
#     loaded_feature_columns=loaded_feature_cols,
#     loaded_product_avg_features_map=loaded_product_avg_map,
#     category='NonExistentCategory'
# )
# print(predictions_not_found_category)
# print(f"Type: {type(predictions_not_found_category)}")


# print("\n--- Flexible Forecasting Demonstration Complete ---")


Loading artifacts from 'Scenario 1'...
Artifacts loaded successfully.

Scenario 2: Predicting for 'Electronics' category for next 2 months (Expected: DataFrame)
Generating total sales value predictions with flexible product filtering...
Total sales value predictions complete.
Shape: (2090, 5), Type: <class 'pandas.core.frame.DataFrame'>


,Category,Sub-Category,Product Name,Forecast_Month,Predicted_Sales
0,Technology,Accessories,AmazonBasics 3-Button USB Wired Mouse,2025-08,26.41
1,Technology,Accessories,AmazonBasics 3-Button USB Wired Mouse,2025-09,25.64
2,Technology,Accessories,AmazonBasics 3-Button USB Wired Mouse,2025-10,23.48
3,Technology,Accessories,AmazonBasics 3-Button USB Wired Mouse,2025-11,26.57
4,Technology,Accessories,AmazonBasics 3-Button USB Wired Mouse,2025-12,23.02
...,...,...,...,...,...
2085,Technology,Phones,netTALK DUO VoIP Telephone Service,2025-08,130.25
2086,Technology,Phones,netTALK DUO VoIP Telephone Service,2025-09,102.93
2087,Technology,Phones,netTALK DUO VoIP Telephone Service,2025-10,104.26
2088,Technology,Phones,netTALK DUO VoIP Telephone Service,2025-11,147.19
